# Results

The first part builds two tables from the raw model output. The activation table holds one row per channel of every activation the model returned, text and each image separately, with a mark for the filter that removes it. The article table holds one row per article with the distinct categories kept, overall and by channel. The second part tests the six hypotheses.

## Main results at a glance (run of 24 September 2026)

Read this first when you come back to the notebook.

* **Corpus.** 48,325 articles in eight outlets. 48,207 have a usable model response. The main analysis uses 44,981 of them, because the 1,255 paid WELT+ previews and the live blogs of elDiario.es (214), Rzeczpospolita (1,437) and Telex (351) are left out. The El Mundo live blogs (280) were already removed from the El Mundo data file, together with the reader comments the scraper had copied into El Mundo texts.
* **Legitimating share** (share of legitimating categories among all categories, counting each category once per article): Poland 13.3%, Germany 13.8%, Spain 18.0%, Hungary 32.4%. Magyar Nemzet 42.3%, Telex 22.5%.
* **H1** supported. **H2** supported. **H3** supported (Magyar Nemzet exceeds Telex by 19.8 points).
* **H4** mixed. Germany counts as equivalent (difference -2.9 points, within the 5-point margin). Spain is undetermined (+3.5 points, interval reaches 5.9). Poland is not supported (-7.8 points, Rzeczpospolita more legitimating), and the gap comes from the different kind of article the two outlets publish, not from their political orientation.
* **H5** supported (+13.7 points more affective categories in images than in text), but the images are mostly read by the model through the article text. **H6** supported only weakly (+2.3 points) and carried by rationalization.
* **Time.** The legitimating share rises from 2022 to 2025 in all four countries. The rise after February 2025 is common to all countries, so it is not a German effect of the new Bundestag.
* **Still to do.** Human validation (precision and recall), which decides how far the image results and the legitimating categories can be trusted.

## Setup

In [ ]:
# imports
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

**What the setup cells do.** They load the libraries, point `final_dir` to the folder with the data files (`<outlet>_final_data_v2.csv`) and the model results (`<outlet>_results_final.csv`), and define the helper functions. The notebook reads those files and never calls the model.

In [ ]:
# folder holding the data files and the results of the annotation run
final_dir = Path(".")

In [ ]:
# read a csv file without pandas turning empty strings into nan
def read_results_csv(path):
    return pd.read_csv(path, keep_default_na=False, na_values=[], low_memory=False)


# parse the stored json of one result row, returning none when it is empty, malformed, or lists categories without their fields
def parse_row(value):
    try:
        parsed = json.loads(value) if value else None
    except (json.JSONDecodeError, TypeError):
        return None
    if isinstance(parsed, dict) and not all(isinstance(e, dict) for e in parsed.get("activated_categories") or []):
        return None
    return parsed


# the image files listed for one article
def list_images(row):
    return [name.strip() for name in str(row["image_filename"]).split(";") if name.strip()]


# the document the model was given and that text evidence is checked against
def coded_document(title, text):
    return str(title) + "\n" + str(text)

**Helper functions.**

* `read_results_csv` reads a csv without turning empty cells into NaN.
* `parse_row` turns the stored model answer (JSON text) into a Python object. It returns None for an empty or broken answer, and also for an answer where the model listed categories as bare strings instead of objects with evidence and role. Such answers are counted as unusable.
* `list_images` lists the image files of an article.
* `coded_document` joins the headline and the body. This is exactly what the model saw, and the text evidence is checked against it.

In [ ]:
# the three filters of section 4.5, identical to the annotation notebook
legit_map = {
    "Authorization": "6.1 Authorization",
    "Moral Evaluation": "6.2 Moral Evaluation",
    "Rationalization": "6.3 Rationalization",
    "Mythopoesis": "6.4 Mythopoesis",
    "Altruism": "6.5 Altruism",
    "Emotion": "6.6 Emotion",
    "Victimization": "6.7 Victimization",
}
delegit_map = {
    "Authorization": "7.1 Authorization",
    "Moral Evaluation": "7.2 Moral Evaluation",
    "Rationalization": "7.3 Rationalization",
    "Mythopoesis": "7.4 Mythopoesis",
    "Emotion": "7.5 Emotion",
    "Demonization and Dehumanization": "7.6 Demonization and Dehumanization",
    "Outcasting": "7.7 Outcasting",
}
required_role = {
    "6.1 Authorization": {"target"},
    "6.2 Moral Evaluation": {"agent"},
    "6.3 Rationalization": {"agent"},
    "6.4 Mythopoesis": {"agent", "topic"},
    "6.5 Altruism": {"provider"},
    "6.6 Emotion": {"topic"},
    "6.7 Victimization": {"target"},
    "7.1 Authorization": {"target"},
    "7.2 Moral Evaluation": {"agent"},
    "7.3 Rationalization": {"agent"},
    "7.4 Mythopoesis": {"agent"},
    "7.5 Emotion": {"agent"},
    "7.6 Demonization and Dehumanization": {"target"},
    "7.7 Outcasting": {"target"},
}


# map a named category label onto its numbered form, returning none when there is no match
def canonical_label(category, orientation):
    category = (category or "").strip()
    if re.match(r"^\d", category):
        return category if category in required_role else None
    return {"legitimation": legit_map, "delegitimation": delegit_map}.get(orientation, {}).get(category)


# normalize whitespace and spacing before punctuation before comparing text
def normalize_for_grounding(text):
    text = re.sub(r"\s+", " ", text).strip()
    return re.sub(r"\s+([.,!?;:])", r"\1", text)


# check that the evidence is verbatim: one exact span, or spans joined by an ellipsis in order
def is_grounded(text_evidence, article_text):
    article = normalize_for_grounding(str(article_text))
    fragments = [f.strip() for f in str(text_evidence).replace("…", "...").split("...") if f.strip()]
    if not fragments:
        return False
    search_from = 0
    for fragment in fragments:
        position = article.find(normalize_for_grounding(fragment), search_from)
        if position == -1:
            return False
        search_from = position + len(normalize_for_grounding(fragment))
    return True

**The three filters of section 4.5.** These are the same rules as in the annotation notebook `9_final_run`.

1. `canonical_label`: if the model wrote a category by name without its number ("Outcasting"), it is mapped to the numbered label ("7.7 Outcasting") within the orientation the model gave. A label that matches no category is discarded.
2. `is_grounded`: the text evidence must appear word for word in the headline or the body, after spaces are normalized. Evidence made of several fragments joined by "..." must match fragment by fragment, in order. This catches quotations the model invented or paraphrased.
3. `required_role`: each category requires Russia in a given role (target, agent, provider, topic). An activation with another role is discarded.

## Article and activation tables

In [ ]:
# split every activation the model returned into one row per channel, keeping every row and marking which filter removes it
countries = {"spiegel": "Germany", "welt": "Germany", "eldiario": "Spain", "elmundo": "Spain",
             "polityka": "Poland", "rp": "Poland", "telex": "Hungary", "magyar": "Hungary"}
telex_live = r"hirfolyam|kozvetitesunk|tudositasunk|percrol-percre"

article_rows, activation_rows = [], []
for source, country in countries.items():
    data = read_results_csv(final_dir / f"{source}_final_data_v2.csv")
    results = read_results_csv(final_dir / f"{source}_results_final.csv")
    raw = dict(zip(results["article_id"], results["parsed"].apply(parse_row)))

    for _, row in data.iterrows():
        url = row["article_url"]
        parsed = raw.get(url)
        usable = isinstance(parsed, dict) and "activated_categories" in parsed
        document = coded_document(row["title"], row["text"])
        kept = {"text": set(), "image": set()}

        for number, e in enumerate(parsed["activated_categories"] if usable else [], start=1):
            label = canonical_label(e.get("category"), e.get("orientation"))
            role_ok = label is not None and e.get("russia_role") in required_role[label]
            base = {"article_url": url, "outlet": source, "country": country, "date": row["date"],
                    "activation": number, "category": label or e.get("category"),
                    "orientation": e.get("orientation"), "russia_role": e.get("russia_role"),
                    "known_category": label is not None, "role_ok": role_ok}

            channels = []
            if e.get("text_evidence"):
                grounded = is_grounded(e["text_evidence"], document)
                channels.append({"channel": "text", "evidence": e["text_evidence"], "image_reason": None,
                                 "grounded": grounded})
            for image in e.get("image_evidence") or []:
                channels.append({"channel": "image", "evidence": image.get("image_file"),
                                 "image_reason": image.get("reason"), "grounded": None})
            if not channels:
                channels.append({"channel": "none", "evidence": None, "image_reason": None, "grounded": None})

            for c in channels:
                keep = label is not None and role_ok and c["channel"] != "none" and c["grounded"] is not False
                activation_rows.append({**base, **c, "kept": keep})
                if keep:
                    kept[c["channel"]].add((label, e.get("orientation")))

        both = kept["text"] | kept["image"]
        count = lambda s, o: sum(orientation == o for _, orientation in s)
        legit, delegit = count(both, "legitimation"), count(both, "delegitimation")
        article_rows.append({
            "article_url": url, "outlet": source, "country": country, "date": row["date"],
            "title": row["title"], "text_chars": len(str(row["text"])), "n_images": len(list_images(row)),
            "usable": usable,
            "paid_preview": source == "welt" and "/plus" in url,
            "live_blog": (source == "eldiario" and "ultima-hora" in url)
                         or (source == "rp" and str(row["title"]).startswith("Wojna Rosji z Ukrainą. Dzień"))
                         or (source == "telex" and bool(re.search(telex_live, url))),
            "n_legitimating": legit, "n_delegitimating": delegit,
            "n_legitimating_text": count(kept["text"], "legitimation"),
            "n_delegitimating_text": count(kept["text"], "delegitimation"),
            "n_legitimating_image": count(kept["image"], "legitimation"),
            "n_delegitimating_image": count(kept["image"], "delegitimation"),
            "score": (legit - delegit) / (legit + delegit) if legit + delegit else None,
        })

articles = pd.DataFrame(article_rows)
activations = pd.DataFrame(activation_rows)
articles.to_csv("corpus_articles.csv", index=False, encoding="utf-8-sig")
activations.to_csv("corpus_activations.csv", index=False, encoding="utf-8-sig")

print(len(articles), "articles |", len(activations), "channel rows |", int(activations["kept"].sum()), "kept")
print()
print(articles.groupby("outlet", sort=False)[["usable", "paid_preview", "live_blog"]].sum().to_string())
print()
print(activations[activations["kept"]].groupby("outlet", sort=False)["channel"].value_counts().unstack().to_string())
print()
removed = activations[~activations["kept"]]
reason = removed.apply(lambda r: "unknown category" if not r["known_category"]
                       else "no evidence" if r["channel"] == "none"
                       else "text not found" if r["grounded"] is False
                       else "wrong role", axis=1)
print(pd.crosstab(removed["channel"], reason.rename("reason")).to_string())

**What this cell does.** It builds the two tables all the analysis uses, directly from the raw model answers.

* The model returns activations. One activation can rest on text evidence, on image evidence, or on both. For the analysis these are separate results, so every activation is split into one row per channel: one `text` row if there is a quotation, and one `image` row for every image the model cites.
* Every row is kept in the table, with marks showing which filter removes it: `known_category`, `role_ok`, `grounded` (text rows only), and `kept` (passes everything). The analysis uses only `kept == True`.
* Grounding applies to text rows only. If the quotation is not in the article, the text row is dropped and the image rows of the same activation stay. Activations with neither a quotation nor an image (`channel == "none"`) are dropped.

**Files written into this folder.**

* `corpus_activations.csv`: one row per channel of each activation. Columns include `category`, `orientation`, `russia_role`, `channel`, `evidence` (quotation, or image file name), `image_reason` (what the model says it saw), and the filter marks.
* `corpus_articles.csv`: one row per article, with the number of distinct legitimating and delegitimating categories kept, overall and per channel, the article `score` = (L - D) / (L + D), and the flags `usable`, `paid_preview`, `live_blog`.

**Result.** 48,325 articles, 48,207 usable. The other 118 are uncoded: 116 unusable answers and 2 Rzeczpospolita articles whose request failed. 160,027 channel rows, 140,334 kept. Removed rows: 18,323 text rows whose quotation was not found (the same number as the grounding filter in `9_final_run`), 1,032 text rows and 150 image rows with the wrong role, 32 rows with an unknown label, and 156 activations with no evidence at all.

## Hypothesis tests

Main sample: articles with a usable response, without the WELT+ previews and the live blogs. Every interval comes from 10,000 resamples of articles within each outlet. A country is the unweighted mean of its two outlets. Shares count distinct categories per article.

In [ ]:
# load the two tables and build, for every article, the counts every hypothesis needs
import numpy as np

articles = pd.read_csv("corpus_articles.csv", low_memory=False)
activations = pd.read_csv("corpus_activations.csv", low_memory=False)
kept = activations[activations["kept"]]

categories = sorted(required_role)
affective = ["6.2 Moral Evaluation", "7.2 Moral Evaluation", "7.6 Demonization and Dehumanization",
             "6.6 Emotion", "7.5 Emotion", "6.7 Victimization"]
propositional = ["6.1 Authorization", "7.1 Authorization", "6.3 Rationalization", "7.3 Rationalization"]
pairs = {"Germany": ("spiegel", "welt"), "Spain": ("eldiario", "elmundo"),
         "Poland": ("polityka", "rp"), "Hungary": ("telex", "magyar")}
outlets = [o for pair in pairs.values() for o in pair]

# one column per category and channel, 1 where the category was kept in that channel of the article
channel_sets = kept.groupby(["article_url", "channel"])["category"].agg(set).unstack()
for channel in ["text", "image"]:
    sets = channel_sets[channel] if channel in channel_sets else pd.Series(dtype=object)
    sets = articles["article_url"].map(sets)
    for c in categories:
        articles[f"{channel}|{c}"] = sets.apply(lambda s: int(isinstance(s, set) and c in s))

text_columns = [f"text|{c}" for c in categories]
image_columns = [f"image|{c}" for c in categories]
articles["both_channels"] = ((articles[text_columns].sum(axis=1) > 0) & (articles[image_columns].sum(axis=1) > 0)).astype(int)
articles["activated"] = (articles["n_legitimating"] + articles["n_delegitimating"] > 0).astype(int)
articles["score_sum"] = articles["score"].fillna(0)
articles["year"] = pd.to_datetime(articles["date"], errors="coerce", utc=True).dt.year

main = articles[articles["usable"] & ~articles["paid_preview"] & ~articles["live_blog"]]
print("articles:", len(articles), "| usable:", int(articles["usable"].sum()), "| main sample:", len(main))
print("dates that could not be read:", int(articles["year"].isna().sum()))
print()
print(main.groupby("outlet", sort=False).agg(
    articles=("article_url", "size"),
    with_activation=("activated", "sum"),
    with_both_channels=("both_channels", "sum")).loc[outlets].to_string())

**What this cell does.** It loads the two tables and adds the columns every test needs.

* `text|<category>` and `image|<category>` are 1 when the category was kept in that channel of the article. Each category counts once per article and channel, so an article with five photographs does not outweigh an article with one.
* `both_channels` is 1 when the article has at least one kept category in the text and at least one in the images.
* `activated` is 1 when the article has at least one category. `score_sum` is the article score, with 0 where there is none.
* `year` is the publication year.

**Main sample.** Usable articles without the WELT+ previews and without the live blogs, 44,981 articles. Per outlet: Spiegel 6,930, Welt 3,030, elDiario.es 1,559, El Mundo 1,738, Polityka 2,158, Rzeczpospolita 13,512, Telex 10,388, Magyar Nemzet 5,666. Articles with categories in both channels, which the H5 and H6 test uses, range from 308 (elDiario.es) to 2,304 (Rzeczpospolita).

In [ ]:
# bootstrap: resample the articles of each outlet with replacement and keep the column sums of every resample
n_boot = 10000
rng = np.random.default_rng(42)
columns = (["n_legitimating", "n_delegitimating", "score_sum", "activated"]
           + text_columns + image_columns
           + [f"both|{c}" for c in text_columns + image_columns])


def outlet_matrix(frame):
    values = frame[["n_legitimating", "n_delegitimating", "score_sum", "activated"] + text_columns + image_columns].to_numpy(float)
    both = frame[text_columns + image_columns].to_numpy(float) * frame[["both_channels"]].to_numpy(float)
    return np.hstack([values, both])


def bootstrap_sums(values):
    n = len(values)
    p = np.full(n, 1 / n)
    parts = []
    for start in range(0, n_boot, 500):
        weights = rng.multinomial(n, p, size=min(500, n_boot - start))
        parts.append(weights @ values)
    return np.vstack(parts)


def run_bootstrap(sample):
    observed, resampled = {}, {}
    for outlet in outlets:
        values = outlet_matrix(sample[sample["outlet"] == outlet])
        observed[outlet] = values.sum(axis=0)
        resampled[outlet] = bootstrap_sums(values)
    return observed, resampled


col = {name: i for i, name in enumerate(columns)}
observed, resampled = run_bootstrap(main)
print("bootstrap done:", n_boot, "resamples per outlet")

**What the bootstrap does, in plain words.** For every outlet it draws 10,000 new samples of the same size from its articles, with replacement, so some articles appear twice and some not at all. Every quantity is recomputed on each sample, and the middle 95% (or 90%) of the 10,000 values is the interval.

* Resampling happens inside each outlet, so each outlet keeps its size and a country keeps both of its outlets.
* Whole articles are resampled, not single activations, because the categories of one article depend on each other.
* Technically the code draws, for each sample, how many times every article is picked (`multinomial`) and multiplies these counts by the article columns. This is the same as drawing articles one by one, only faster.
* `rng = np.random.default_rng(42)` fixes the random numbers, so running the notebook again gives the same intervals.

In [ ]:
# helpers that turn column sums into the quantities of the hypotheses, for the observed data and for every resample
def proportion(sums):
    legit, delegit = sums[..., col["n_legitimating"]], sums[..., col["n_delegitimating"]]
    return legit / (legit + delegit)


def mean_score(sums):
    return sums[..., col["score_sum"]] / sums[..., col["activated"]]


def by_outlet(function, obs, res):
    return {o: (function(obs[o]), function(res[o])) for o in outlets}


def by_country(outlet_values):
    return {country: ((outlet_values[a][0] + outlet_values[b][0]) / 2, (outlet_values[a][1] + outlet_values[b][1]) / 2)
            for country, (a, b) in pairs.items()}


def interval(draws, level=0.95):
    tail = (1 - level) / 2 * 100
    return np.nanpercentile(draws, tail), np.nanpercentile(draws, 100 - tail)


def row(label, estimate, draws, level=0.95, scale=100):
    low, high = interval(draws, level)
    return {"": label, "estimate": round(estimate * scale, 2), "low": round(low * scale, 2), "high": round(high * scale, 2)}


def bootstrap_p(estimate, draws):
    opposite = np.mean(draws <= 0) if estimate > 0 else np.mean(draws >= 0)
    return min(1.0, 2 * opposite)


def holm(p_values):
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values))
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p_values) - rank) * p_values[i])
        adjusted[i] = min(1.0, running)
    return adjusted

**Helper functions for the tests.**

* `proportion`: legitimating share of an outlet = sum of distinct legitimating categories over all its articles / sum of distinct legitimating and delegitimating categories. An article with six categories weighs more than one with two, as in section 4.6.
* `mean_score`: mean of the article scores, where every article with a category weighs the same. It is the robustness check reported next to every proportion.
* `by_country`: a country is the unweighted mean of its two outlets, so Rzeczpospolita does not dominate Poland and Telex does not dominate Hungary.
* `interval`: percentile interval from the 10,000 bootstrap values.
* `bootstrap_p`: two times the share of bootstrap values on the other side of zero from the observed difference.
* `holm`: Holm correction for the fourteen per-category tests.

In [ ]:
# descriptive quantities: proportion of legitimating activations and mean article score by outlet and by country
def describe(obs, res):
    props = by_outlet(proportion, obs, res)
    scores = by_outlet(mean_score, obs, res)
    rows = []
    for o in outlets:
        rows.append({**row(o, *props[o]), **{f"score {k}": v for k, v in row(o, *scores[o], scale=1).items() if k}})
    for country, (estimate, draws) in by_country(props).items():
        score_estimate, score_draws = by_country(scores)[country]
        rows.append({**row(country, estimate, draws),
                     **{f"score {k}": v for k, v in row(country, score_estimate, score_draws, scale=1).items() if k}})
    return pd.DataFrame(rows).set_index("")


print("legitimating share of activations (%) and mean article score, 95% intervals")
print(describe(observed, resampled).to_string())

**Result: legitimating share and mean article score, main sample.**

**How to read it.** In every outlet delegitimation dominates, since the share is below 50% and the scores are negative. Magyar Nemzet is far above everyone else. The mean scores give the same order as the shares, so the results are not driven by a few densely coded articles.

In [ ]:
# h1: the country proportion of legitimating activations lies below one half in poland, germany and spain
country_props = by_country(by_outlet(proportion, observed, resampled))
h1 = pd.DataFrame([row(c, *country_props[c]) for c in ["Poland", "Germany", "Spain"]]).set_index("")
h1["supported"] = h1["high"] < 50
print("H1, legitimating share (%), 95% interval")
print(h1.to_string())
print()

# h2: hungary minus each of the other three countries
h2 = pd.DataFrame([row(f"Hungary - {c}", country_props["Hungary"][0] - country_props[c][0],
                       country_props["Hungary"][1] - country_props[c][1])
                   for c in ["Poland", "Germany", "Spain"]]).set_index("")
h2["excludes zero"] = (h2["low"] > 0) | (h2["high"] < 0)
print("H2, difference in percentage points, 95% interval")
print(h2.to_string())
print("supported:", bool((h2["estimate"] > 0).all() and (h2["low"] > 0).all()))

**H1 (Poland, Germany, Spain: delegitimation predominates).** Supported for all three. The upper ends of the intervals are 14.0%, 14.6% and 19.4%, far below 50%. H1 is a low bar: it fixes the baseline, and the informative tests are H2 to H4.

**H2 (Hungary has a higher legitimating share than the other countries).** Supported. Hungary exceeds Poland by 19.1 points [18.0, 20.2], Germany by 18.6 [17.5, 19.8] and Spain by 14.4 [12.8, 16.0]. All three intervals lie above zero. Hungary is also the highest country in every single year (see the time check at the end).

In [ ]:
# h3: magyar nemzet minus telex, with a bootstrap interval and a permutation test
outlet_props = by_outlet(proportion, observed, resampled)
h3_estimate = outlet_props["magyar"][0] - outlet_props["telex"][0]
h3_draws = outlet_props["magyar"][1] - outlet_props["telex"][1]

hungary = main[main["outlet"].isin(["telex", "magyar"])]
legit = hungary["n_legitimating"].to_numpy(float)
total = legit + hungary["n_delegitimating"].to_numpy(float)
n_magyar = int((hungary["outlet"] == "magyar").sum())
permuted = np.empty(n_boot)
for i in range(n_boot):
    order = rng.permutation(len(hungary))
    a, b = order[:n_magyar], order[n_magyar:]
    permuted[i] = legit[a].sum() / total[a].sum() - legit[b].sum() / total[b].sum()
p_permutation = np.mean(np.abs(permuted) >= abs(h3_estimate))

h3 = pd.DataFrame([row("Magyar Nemzet - Telex", h3_estimate, h3_draws)]).set_index("")
h3["permutation p"] = p_permutation
print("H3, difference in percentage points, 95% interval")
print(h3.to_string())
print("supported:", bool(h3_estimate > 0 and interval(h3_draws)[0] > 0))

**H3 (Magyar Nemzet more legitimating than Telex).** Supported. The difference is +19.8 points [18.1, 21.4].

The permutation test shuffles the outlet labels across all Hungarian articles 10,000 times and recomputes the difference each time. None of the 10,000 shuffled differences was as large as the observed one, so p < 0.0001 (the cell prints 0.0, which means "smaller than 1 in 10,000", not exactly zero). The difference holds in every year, between 9 and 19 points.

In [ ]:
# h4: the two outlets of poland, germany and spain differ by less than 5 points, read from a 90% interval
margin = 0.05
h4_rows = []
for country in ["Poland", "Germany", "Spain"]:
    left, right = pairs[country]
    estimate = outlet_props[left][0] - outlet_props[right][0]
    draws = outlet_props[left][1] - outlet_props[right][1]
    h4_rows.append(row(f"{country}: {left} - {right}", estimate, draws, level=0.90))
h4 = pd.DataFrame(h4_rows).set_index("")
h4["within 5 points"] = (h4["low"] > -margin * 100) & (h4["high"] < margin * 100)
print("H4, difference in percentage points, 90% interval, equivalence margin 5 points")
print(h4.to_string())

**H4 (the two outlets of Poland, Germany and Spain do not differ).** Tested as equivalence: the 90% interval of the difference must lie entirely within -5 and +5 points. The difference is always the left-leaning outlet minus the right-leaning one.

| pair | difference | 90% interval | verdict |
|---|---|---|---|
| Germany, Spiegel - Welt | -2.86 | -4.16 to -1.59 | equivalent |
| Spain, elDiario.es - El Mundo | +3.45 | 1.11 to 5.85 | undetermined |
| Poland, Polityka - Rzeczpospolita | -7.83 | -9.00 to -6.64 | not equivalent |

**How to read it.**

* Germany: Welt is slightly more legitimating and the difference is not zero, but it is smaller than the 5-point margin, so by the rule fixed in 4.8 the two outlets count as equivalent. After February 2025 the difference disappears.
* Spain: the data rule out differences above 5.9 points but not those between 5 and 5.9, so the result is undetermined. The yearly check shows the two outlets drift apart from 2023, with elDiario.es rising.
* Poland: Rzeczpospolita is almost twice as legitimating as Polityka, in every year. The length check below shows that among the longest articles (over 8,000 characters) the two outlets do not differ (+1.2 points). The gap comes from the short news items of the daily Rzeczpospolita, which the weekly Polityka does not publish. It is a difference of genre, not of orientation.

In [ ]:
# h5 and h6: share of affective and of propositional categories in each channel, articles where both channels are active
def channel_share(sums, group, channel, prefix="both|"):
    group_sum = sum(sums[..., col[f"{prefix}{channel}|{c}"]] for c in group)
    all_sum = sum(sums[..., col[f"{prefix}{channel}|{c}"]] for c in categories)
    return group_sum / all_sum


def pooled(obs, res):
    return sum(obs[o] for o in outlets), sum(res[o] for o in outlets)


def channel_tests(obs, res, prefix):
    total_obs, total_res = pooled(obs, res)
    rows = []
    for label, group, first, second in [("H5 affective, image - text", affective, "image", "text"),
                                        ("H6 propositional, text - image", propositional, "text", "image")]:
        estimate = channel_share(total_obs, group, first, prefix) - channel_share(total_obs, group, second, prefix)
        draws = channel_share(total_res, group, first, prefix) - channel_share(total_res, group, second, prefix)
        rows.append({**row(label, estimate, draws),
                     f"{first} share %": round(channel_share(total_obs, group, first, prefix) * 100, 2),
                     f"{second} share %": round(channel_share(total_obs, group, second, prefix) * 100, 2)})
    table = pd.DataFrame(rows).set_index("")
    table["supported"] = table["low"] > 0
    return table


print("H5 and H6, articles with at least one category in each channel, difference in points, 95% interval")
print(channel_tests(observed, resampled, "both|").to_string())
print()

# the difference between channels for each of the fourteen categories, p-values adjusted by holm
total_obs, total_res = pooled(observed, resampled)
per_category = []
for c in categories:
    estimate = channel_share(total_obs, [c], "image") - channel_share(total_obs, [c], "text")
    draws = channel_share(total_res, [c], "image") - channel_share(total_res, [c], "text")
    per_category.append({**row(c, estimate, draws), "p": bootstrap_p(estimate, draws)})
per_category = pd.DataFrame(per_category).set_index("")
per_category["p holm"] = holm(per_category["p"].to_numpy()).round(4)
print("share of each category among image categories minus among text categories, points, 95% interval")
print(per_category.to_string())

**H5 and H6 (the division of work between images and text).** Computed on the 9,039 articles that have at least one category in the text and at least one in the images, so the same articles are compared across the two channels.

* Affective group: moral evaluation (6.2, 7.2), demonization and dehumanization (7.6), emotion (6.6, 7.5), victimization (6.7).
* Propositional group: authorization (6.1, 7.1), rationalization (6.3, 7.3).
* A share is the number of group categories in a channel divided by all categories in that channel.

**Result.**

* H5 supported. Affective categories make up 56.4% of image categories and 42.7% of text categories, a difference of +13.7 points [13.1, 14.3].
* H6 supported, but weakly. Propositional categories make up 27.6% of text categories and 25.3% of image categories, +2.3 points [1.8, 2.8].

**Per category** (image share minus text share, all significant after Holm). More frequent in images: delegitimating emotion 7.5 (+7.7), delegitimating moral evaluation 7.2 (+7.6), legitimating authorization 6.1 (+6.8). More frequent in text: outcasting 7.7 (-8.4), legitimating rationalization 6.3 (-4.8), delegitimating rationalization 7.3 (-2.9).

**Careful.** H6 rests on rationalization. Authorization splits between the channels: legitimating authorization (6.1) is more frequent in images, because official photographs of Putin with flags and state insignia are coded as authorization. The check on the image channel further down shows that most of the affective load of images comes from images the model reads through the article text.

## Robustness checks

In [ ]:
# robustness: h1 to h4 with the paid previews and live blogs kept, and h5 and h6 on every article with a category in either channel
full = articles[articles["usable"]]
observed_full, resampled_full = run_bootstrap(full)
print("all usable articles, paid previews and live blogs included")
print(describe(observed_full, resampled_full).to_string())
print()
print("H5 and H6 on every article, not only those with both channels active")
print(channel_tests(observed, resampled, "").to_string())
print()

# robustness: germany before and after the february 2025 election
cutoff = pd.Timestamp("2025-02-23", tz="UTC")
germany_dates = pd.to_datetime(main["date"], errors="coerce", utc=True)
for label, part in [("before 23 February 2025", main[germany_dates < cutoff]), ("from 23 February 2025", main[germany_dates >= cutoff])]:
    part = part[part["outlet"].isin(["spiegel", "welt"])]
    sums = {o: outlet_matrix(part[part["outlet"] == o]).sum(axis=0) for o in ["spiegel", "welt"]}
    spiegel, welt = proportion(sums["spiegel"]), proportion(sums["welt"])
    print(f"Germany {label}: articles {len(part)} | Spiegel {spiegel * 100:.2f}% | Welt {welt * 100:.2f}% "
          f"| country {(spiegel + welt) / 2 * 100:.2f}% | difference {(spiegel - welt) * 100:.2f} points")
print()

# robustness: the difference within each pair, year by year, and how the outlets of a pair differ in what they publish
year_rows = []
for country, (a, b) in pairs.items():
    for year, part in main[main["outlet"].isin([a, b])].groupby("year"):
        sums = {o: outlet_matrix(part[part["outlet"] == o]).sum(axis=0) for o in (a, b)}
        year_rows.append({"country": country, "year": int(year),
                          "difference": round((proportion(sums[a]) - proportion(sums[b])) * 100, 2)})
print("difference in points by year, first outlet minus second: spiegel - welt, eldiario - elmundo, polityka - rp, telex - magyar")
print(pd.DataFrame(year_rows).pivot(index="year", columns="country", values="difference").to_string())
print()
print(main.groupby("outlet").agg(
    median_chars=("text_chars", "median"),
    with_image_pct=("n_images", lambda s: round((s > 0).mean() * 100, 1)),
    first_year=("year", "min"), last_year=("year", "max")).loc[outlets].to_string())

**Robustness checks and their results.**

1. **Paid previews and live blogs kept.** H1 to H3 do not change. Two pairs move: Poland shrinks from -7.8 to -5.5 points (the Rzeczpospolita live blogs are strongly delegitimating) and Spain falls from +3.5 to +0.8 points.
2. **H5 and H6 on every article**, not only those with both channels active: the effects are larger, +17.5 and +4.6 points.
3. **Germany before and after 23 February 2025.** Spiegel rises from 11.2% to 17.6%, Welt from 14.5% to 17.4%, and the difference between them disappears. The time check at the end shows the same rise in all four countries, so it is not caused by the new Bundestag.
4. **Difference within each pair by year.** Poland: -7 to -10 points every year. Germany: -2 to -5 points until 2024, around 0 afterwards. Spain: -2.4 in 2022, then +7 to +15 from 2023. Hungary: large every year.
5. **What the outlets publish.** Median article length in characters: elDiario.es 912, Telex 1,117, Magyar Nemzet 2,112, Spiegel 2,570, Welt 3,516, Polityka 3,794, El Mundo 4,615, Rzeczpospolita 5,891 (the Rzeczpospolita texts still contain the site menu and footer, so its real articles are shorter). Between 86% and 100% of articles carry at least one image.

The warnings `invalid value encountered in divide` are harmless: in some year or length band an outlet has no activation at all, so its share is empty.

## Checks on the pair differences and on the image channel

In [ ]:
# robustness: the difference within each pair of outlets inside bands of article length
bands = [0, 1000, 2000, 4000, 8000, 10**7]
band_labels = ["<1000", "1000-2000", "2000-4000", "4000-8000", ">8000"]
main_banded = main.assign(band=pd.cut(main["text_chars"], bands, labels=band_labels))

band_rows = []
for country, (a, b) in pairs.items():
    for band, part in main_banded[main_banded["outlet"].isin([a, b])].groupby("band", observed=True):
        sums = {o: outlet_matrix(part[part["outlet"] == o]).sum(axis=0) for o in (a, b)}
        band_rows.append({"country": country, "band": band,
                          f"n {a}": int((part["outlet"] == a).sum()), f"n {b}": int((part["outlet"] == b).sum()),
                          "difference": round((proportion(sums[a]) - proportion(sums[b])) * 100, 2)})
print("difference in points within length bands, first outlet minus second")
for country in pairs:
    print(pd.DataFrame([r for r in band_rows if r["country"] == country]).drop(columns="country").to_string(index=False))
    print()

**Difference within each pair inside bands of article length.** Checks whether a difference between two outlets is only a difference in how long their articles are.

* Poland: the two outlets hardly overlap in length. Rzeczpospolita has 11,278 of its articles between 4,000 and 8,000 characters, Polityka has most of its articles between 2,000 and 4,000 and above 8,000. Where both are well represented, above 8,000 characters (Polityka 549, Rzeczpospolita 2,119), the difference is only +1.2 points. The Polish gap is carried by the ordinary news items of Rzeczpospolita.
* Germany: Welt is more legitimating in every band, by 2.5 to 9 points. Length does not explain the German difference.
* Spain: the outlets overlap little (most elDiario.es articles are under 1,000 characters), so the bands say little.
* Hungary: the gap is large in every band.

In [ ]:
# how much of the kept text evidence is reported speech, marked by quotation marks or a verb of saying
speech = re.compile(
    r"[\"«»„“”]"
    r"|\b(?:sagte|sagt|erklärte|betonte|behauptete|warnte|zufolge|laut)\b"
    r"|\b(?:dijo|afirmó|aseguró|declaró|señaló|añadió|explicó|advirtió|según)\b"
    r"|\b(?:powiedział|stwierdził|oświadczył|zapowiedział|podkreślił|przekonywał|dodał|zaznaczył|według)\b"
    r"|\b(?:mondta|közölte|kijelentette|hangsúlyozta|állította|nyilatkozta|fogalmazott|szerint)\b",
    re.IGNORECASE)

text_kept = kept[kept["channel"] == "text"].copy()
text_kept = text_kept[text_kept["article_url"].isin(set(main["article_url"]))]
text_kept["reported_speech"] = text_kept["evidence"].astype(str).str.contains(speech)

print("share of kept text activations that are reported speech (%)")
print((text_kept.groupby(["outlet", "orientation"])["reported_speech"].mean().unstack() * 100).round(1).loc[outlets].to_string())
print()
for orientation in ["legitimation", "delegitimation"]:
    for flag in [True, False]:
        sample = text_kept[(text_kept["orientation"] == orientation) & (text_kept["reported_speech"] == flag)]
        print(orientation, "| reported speech" if flag else "| no reported speech")
        for _, r in sample.sample(min(4, len(sample)), random_state=3).iterrows():
            print(f'   {r["outlet"]} | {r["category"]} | {str(r["evidence"])[:140]}')
    print()

**How much of the evidence is reported speech.** A text row counts as reported speech when its quotation contains quotation marks or a verb of saying in one of the four languages (dijo, sagte, powiedział, mondta, según, szerint and others). The rule is approximate: it also catches quotations of Ukrainian or Western sources and quotation marks around single words.

**Result.** In every outlet legitimating evidence is more often reported speech than delegitimating evidence (for example Rzeczpospolita 35% against 19%, Polityka 29% against 16%). But only between 23% and 45% of legitimating evidence is reported speech, so legitimation is not mostly quoted Kremlin statements. Much of the rest reports Russian acts, such as an announced ceasefire or a commemoration ceremony.

In [ ]:
# the legitimating share by outlet when text evidence that is reported speech is left out, images kept
def outlet_share_from_rows(rows):
    distinct = rows.drop_duplicates(["article_url", "category"])
    legit = distinct.groupby("outlet")["orientation"].apply(lambda s: (s == "legitimation").sum())
    total = distinct.groupby("outlet").size()
    return legit / total * 100


main_kept = kept[kept["article_url"].isin(set(main["article_url"]))]
speech_urls_rows = main_kept["channel"].eq("text") & main_kept["evidence"].astype(str).str.contains(speech)
shares = pd.DataFrame({"all evidence %": outlet_share_from_rows(main_kept),
                       "without reported speech %": outlet_share_from_rows(main_kept[~speech_urls_rows])}).loc[outlets].round(2)
print(shares.to_string())
print()
for country, (a, b) in pairs.items():
    print(f"{country}: {a} - {b} | all evidence {shares.loc[a].iloc[0] - shares.loc[b].iloc[0]:.2f} points"
          f" | without reported speech {shares.loc[a].iloc[1] - shares.loc[b].iloc[1]:.2f} points")

**Legitimating share when reported speech is left out.** Recomputes each outlet's share without the text rows marked as reported speech, keeping the image rows.

**Result.** All shares fall a little, by 0.3 to 2.5 points. The Polish gap narrows only from -7.8 to -6.6 points, so reported speech does not explain it. The Hungarian gap stays at about 20 points.

In [ ]:
# robustness for h5 and h6: image evidence whose stated reason leans on the article text is left out
leans_on_text = re.compile(r"\b(?:text|article|headline|caption|context|described|mentioned|reported|according)\b", re.IGNORECASE)
image_rows = main_kept[main_kept["channel"] == "image"]
image_alone = image_rows[~image_rows["image_reason"].astype(str).str.contains(leans_on_text)]
print("kept image rows:", len(image_rows), "| reason does not refer to the text:", len(image_alone),
      f"({len(image_alone) / len(image_rows) * 100:.1f}%)")


def channel_group_shares(text_rows, image_rows_used):
    text_sets = text_rows.groupby("article_url")["category"].agg(set)
    image_sets = image_rows_used.groupby("article_url")["category"].agg(set)
    both = text_sets.index.intersection(image_sets.index)
    result = {}
    for channel, sets in [("text", text_sets.loc[both]), ("image", image_sets.loc[both])]:
        flat = [c for s in sets for c in s]
        result[channel] = {"affective": sum(c in affective for c in flat) / len(flat) * 100,
                           "propositional": sum(c in propositional for c in flat) / len(flat) * 100}
    return len(both), result


text_rows = main_kept[main_kept["channel"] == "text"]
for label, used in [("all image rows", image_rows), ("image rows whose reason does not refer to the text", image_alone)]:
    n, s = channel_group_shares(text_rows, used)
    print(f"{label}: articles with both channels {n}")
    print(f"   affective      image {s['image']['affective']:.2f}% | text {s['text']['affective']:.2f}% | difference {s['image']['affective'] - s['text']['affective']:.2f}")
    print(f"   propositional  text {s['text']['propositional']:.2f}% | image {s['image']['propositional']:.2f}% | difference {s['text']['propositional'] - s['image']['propositional']:.2f}")

**Do the images carry the categories themselves?** For each image row the model writes what it saw (`image_reason`). When that explanation refers to the article (words such as text, article, caption, context, described, mentioned), the model relied on the text to code the image. This cell recomputes H5 and H6 using only image rows whose explanation does not refer to the text.

**Result.**

* Only 4,683 of 17,119 kept image rows (27.4%) have an explanation that does not refer to the text.
* On those images alone, H5 reverses: affective categories are 30.9% of image categories against 33.3% of text categories (-2.4 points). H6 also reverses: propositional categories are 45.1% of image categories against 36.0% of text categories, mostly because official photographs are coded as legitimating authorization.

**How to read it.** The affective load of images is produced mostly through the text. A photograph of a destroyed building becomes delegitimation of Russia because the article names Russia as the cause. Images judged on what they show alone mostly carry authorization. This fits anchorage (Barthes) as set out in 2.5. The rule is approximate (see the next cell), so the 27.4% figure should not be reported as exact, but the direction of the result is clear.

In [ ]:
# show image reasons that were read as leaning on the text and reasons that were not, to check the rule by eye
image_rows = main_kept[main_kept["channel"] == "image"].copy()
image_rows["leans_on_text"] = image_rows["image_reason"].astype(str).str.contains(leans_on_text)
for flag in [True, False]:
    print("leans on the text" if flag else "does not lean on the text")
    for _, r in image_rows[image_rows["leans_on_text"] == flag].sample(10, random_state=5).iterrows():
        print(f'   {r["category"]} | {str(r["image_reason"])[:220]}')
    print()

**Checking the rule by eye.** Ten explanations from each group.

* Rows marked as leaning on the text mostly do ("text explicitly attributes bombing to Russian forces", "the context of the article attributes the destruction to Russia"). A few are false positives, for instance text written on a poster in the image, or a person "identifiable by his appearance and context".
* Rows marked as not leaning on the text describe only what is visible: Putin in front of the Russian flag, police with visible insignia detaining a civilian, a caricature with the Z symbol, a medal ceremony in front of the state coat of arms.

The rule is good enough to show the direction of the result, not to give an exact share.

## Time check

In [ ]:
# legitimating share by year for every outlet and every country, and before and after 23 february 2025
def share_table(frame, period_column):
    rows = []
    for period, part in frame.groupby(period_column):
        entry = {period_column: period}
        for country, (a, b) in pairs.items():
            shares = [proportion(outlet_matrix(part[part["outlet"] == o]).sum(axis=0)) * 100 for o in (a, b)]
            entry[a], entry[b] = round(shares[0], 2), round(shares[1], 2)
            entry[country] = round((shares[0] + shares[1]) / 2, 2)
        rows.append(entry)
    return pd.DataFrame(rows).set_index(period_column)


by_period = main.assign(period=np.where(pd.to_datetime(main["date"], errors="coerce", utc=True)
                                        < pd.Timestamp("2025-02-23", tz="UTC"),
                                        "1 before 23 Feb 2025", "2 from 23 Feb 2025"))

print("legitimating share (%) by year, countries")
print(share_table(main, "year")[list(pairs)].to_string())
print()
print("legitimating share (%) by year, outlets")
print(share_table(main, "year")[outlets].to_string())
print()
print("legitimating share (%) before and after 23 February 2025")
print(share_table(by_period, "period")[list(pairs) + outlets].T.to_string())
print()
print("articles by year and outlet")
print(main.groupby(["year", "outlet"]).size().unstack()[outlets].to_string())

**Time check: legitimating share by year.** Shows whether the rise after February 2025 seen in Germany is specific to Germany.

**Result.**

| | 2022 | 2023 | 2024 | 2025 | 2026 (half year) |
|---|---|---|---|---|---|
| Germany | 10.3 | 13.7 | 15.8 | 16.8 | 19.4 |
| Spain | 12.9 | 19.7 | 21.2 | 26.6 | 24.9 |
| Poland | 10.9 | 14.9 | 13.7 | 16.0 | 12.7 |
| Hungary | 26.4 | 34.3 | 36.8 | 39.8 | 29.0 |

Before and after 23 February 2025: Germany 12.8 to 17.5, Spain 15.7 to 25.5, Poland 12.8 to 15.1, Hungary 31.4 to 36.7.

**How to read it.**

* The legitimating share rises from 2022 to 2025 in all four countries. In 2022 coverage was dominated by the invasion itself and was almost entirely condemning; by 2025 negotiations and Russian diplomacy fill the news.
* The rise after February 2025 happens everywhere, so it is not an effect of the new German parliament.
* 2026 covers only the first half of the year and few articles (88 to 502 per outlet), so it should not be interpreted.
* Attention falls sharply: Rzeczpospolita published 6,000 Russia-central articles in 2022 and 1,489 in 2025, Telex 5,315 and 1,215.
* Outlets differ in how their articles spread over the years (51% of Telex articles are from 2022, 22% of Magyar Nemzet articles), but H2, H3 and the Polish H4 result hold in every single year, so the year mix does not drive them.

## Descriptive statistics

In [ ]:
# descriptive statistics for section 5.1, all on the main sample
kept_main = kept[kept["article_url"].isin(set(main["article_url"]))]
activation_counts = kept_main.drop_duplicates(["article_url", "activation"]).groupby("article_url").size()
distinct_counts = kept_main.drop_duplicates(["article_url", "category"]).groupby("article_url").size()
described = main.assign(
    activations=main["article_url"].map(activation_counts).fillna(0),
    distinct=main["article_url"].map(distinct_counts).fillna(0))

# corpus composition and density of coding by outlet
composition = described.groupby("outlet").agg(
    articles=("article_url", "size"),
    median_chars=("text_chars", "median"),
    with_image_pct=("n_images", lambda s: (s > 0).mean() * 100),
    median_images=("n_images", "median"),
    legit_per_article=("n_legitimating", "mean"),
    legit_sd=("n_legitimating", "std"),
    delegit_per_article=("n_delegitimating", "mean"),
    delegit_sd=("n_delegitimating", "std"),
    activations_per_article=("activations", "mean"),
    no_activation_pct=("activated", lambda s: (1 - s.mean()) * 100))
composition["activations_per_distinct_category"] = (described.groupby("outlet")["activations"].sum()
                                                     / described.groupby("outlet")["distinct"].sum())
print("corpus composition and density of coding by outlet")
print(composition.loc[outlets].round(2).to_string())
print()

# attention: articles and share without any activation, by year
print("articles by year")
print(main.groupby(["year", "outlet"]).size().unstack()[outlets].to_string())
print()
print("share of articles without any activation (%) by year")
print((1 - main.groupby(["year", "outlet"])["activated"].mean().unstack()[outlets]).mul(100).round(1).to_string())
print()

# category activation frequencies: share of articles activating each category, by outlet
category_share = pd.DataFrame({o: [main.loc[main["outlet"] == o, [f"text|{c}", f"image|{c}"]].max(axis=1).mean() * 100
                                   for c in categories] for o in outlets}, index=categories)
category_share.round(1).to_csv("category_share_by_outlet.csv")
print("share of articles activating each category (%), text or image")
print(category_share.round(1).to_string())
print()

# distribution of the article score by outlet and country
def score_split(frame):
    scored = frame["score"]
    return pd.Series({"below zero %": (scored < 0).mean() * 100, "zero %": (scored == 0).mean() * 100,
                      "above zero %": (scored > 0).mean() * 100, "no activation %": scored.isna().mean() * 100})


split = pd.DataFrame({o: score_split(main[main["outlet"] == o]) for o in outlets}).T
for country, (a, b) in pairs.items():
    split.loc[country] = (split.loc[a] + split.loc[b]) / 2
print("distribution of article scores (% of articles); a country is the mean of its two outlets")
print(split.round(1).to_string())
print()

# monthly volume by country for figure 1
monthly = main.assign(month=pd.to_datetime(main["date"], errors="coerce", utc=True).dt.strftime("%Y-%m"))
monthly.groupby(["month", "country"]).size().unstack().fillna(0).astype(int).to_csv("monthly_volume_by_country.csv")
print("saved: category_share_by_outlet.csv, monthly_volume_by_country.csv")

**Descriptive statistics for section 5.1 (main sample, 44,981 articles).** The cell describes the outlets on four things: what they publish, how densely their articles are coded, which categories they activate, and how their article scores are distributed. It also saves two files for figures, `category_share_by_outlet.csv` (heat map of categories by outlet) and `monthly_volume_by_country.csv` (monthly volume by country).

**1. Corpus composition and density of coding.**

| outlet | articles | median chars | with image | legit per article (sd) | delegit per article (sd) | activations per article | no activation |
|---|---|---|---|---|---|---|---|
| Spiegel | 6,930 | 2,570 | 97.9% | 0.30 (0.84) | 2.12 (1.51) | 2.84 | 18.2% |
| Welt | 3,030 | 3,516 | 89.6% | 0.34 (0.90) | 1.89 (1.41) | 2.55 | 18.0% |
| elDiario.es | 1,559 | 912 | 98.7% | 0.32 (0.78) | 1.30 (1.30) | 1.74 | 28.2% |
| El Mundo | 1,738 | 4,615 | 95.9% | 0.39 (0.97) | 2.00 (1.49) | 2.84 | 15.7% |
| Polityka | 2,158 | 3,794 | 98.8% | 0.23 (0.73) | 2.21 (1.54) | 2.77 | 15.1% |
| Rzeczpospolita | 13,512 | 5,891 | 94.3% | 0.33 (0.87) | 1.61 (1.43) | 2.20 | 24.4% |
| Telex | 10,388 | 1,117 | 100.0% | 0.36 (0.91) | 1.23 (1.37) | 1.79 | 35.4% |
| Magyar Nemzet | 5,666 | 2,112 | 86.0% | 0.93 (1.44) | 1.27 (1.37) | 2.53 | 22.0% |

How to read it:
* Almost every article carries at least one image, from 86% at Magyar Nemzet to 100% at Telex, so the outlets are comparable in the amount of visual material. The median number of images per article is 1 to 3.
* Article length varies a lot: elDiario.es and Telex publish short items (median about 900 and 1,100 characters), Rzeczpospolita long ones. The Rzeczpospolita texts still contain the site menu and footer, so its real articles are shorter than 5,891 characters.
* A typical article activates about two delegitimating categories and a fraction of one legitimating category. Magyar Nemzet is the exception, with almost one legitimating category per article (0.93) and fewer delegitimating ones (1.27).
* The standard deviations are large, so articles differ a lot within every outlet.
* Activations per distinct category are 1.08 to 1.19 everywhere: a category is usually activated once in an article, rarely repeated on separate evidence. Counting distinct categories or all activations therefore gives almost the same picture.
* The share of articles with no activation ranges from 15.1% (Polityka) to 35.4% (Telex). It follows article length, since short items give the categories little room. This share cannot be compared across outlets as a measure of how neutral they are.

**2. Attention over time (articles per year).** Volume falls sharply after 2022 in most outlets: Rzeczpospolita from 6,000 to 1,489 articles in 2025, Telex from 5,315 to 1,215, Spiegel from 2,684 to 1,135, elDiario.es from 724 to 358. Magyar Nemzet is the exception and stays at about 1,100 to 1,500 articles a year. 2026 covers only the first half of the year. The share of articles with no activation rises a little over time in several outlets (elDiario.es from 22% in 2022 to 37% in 2025, Spiegel from 18% to 20%).

**3. Share of articles activating each category (text or image).**
* Outcasting (7.7) and delegitimating moral evaluation (7.2) are the two most frequent categories in every outlet, in 37% to 64% of articles. Spiegel has the highest shares of both (64.4% and 59.8%).
* Polityka stands out on delegitimating rationalization (7.3), in 39.9% of its articles against 8% to 19% elsewhere. Its analytical articles often present Russian conduct as failing its own goals.
* El Mundo has the highest share of demonization and dehumanization (7.6, 13.4%), Spiegel and El Mundo the highest share of delegitimating emotion (7.5, about 28.5%).
* Magyar Nemzet activates every legitimating category three to four times as often as the other outlets: authorization 6.1 in 28.3% of articles, rationalization 6.3 in 22.5%, mythopoesis 6.4 in 18.8%. Telex is slightly above the non-Hungarian outlets on 6.1 (12.4%) and 6.3 (8.1%).
* The short-item outlets (elDiario.es, Telex) and Magyar Nemzet show the lowest shares of the frequent delegitimating categories, partly because their articles are shorter.

**4. Distribution of article scores (% of articles).**

| | below zero | exactly zero | above zero | no activation |
|---|---|---|---|---|
| Germany | 69.5 | 2.4 | 10.1 | 18.1 |
| Spain | 62.4 | 2.5 | 13.2 | 21.9 |
| Poland | 67.8 | 1.7 | 10.8 | 19.7 |
| Hungary | 45.6 | 2.6 | 23.1 | 28.7 |

* In Germany, Spain and Poland about two thirds of articles lean delegitimating and about one in ten leans legitimating.
* Magyar Nemzet is the only outlet where almost a third of articles lean legitimating (31.9%, against 42.7% leaning delegitimating).
* Articles with an exact balance are rare everywhere (1.7% to 3.4%), so articles almost always lean one way.
* A country here is the mean of its two outlets, as in the tests.

In [ ]:
# composition of legitimating and delegitimating content by outlet and country, and correspondence analysis of outlets by categories, main sample
kept_main = activations[activations["kept"] & activations["article_url"].isin(main["article_url"])]
distinct = kept_main.drop_duplicates(["article_url", "category"])
counts = pd.crosstab(distinct["outlet"], distinct["category"]).loc[outlets]

for side in ["7", "6"]:
    part = counts[[c for c in counts.columns if c.startswith(side)]]
    composition = part.div(part.sum(axis=1), axis=0) * 100
    for country, (a, b) in pairs.items():
        composition.loc[country] = (composition.loc[a] + composition.loc[b]) / 2
    print(composition.round(1).T.to_string())
    print()

p = counts / counts.values.sum()
r, c = p.sum(axis=1).values, p.sum(axis=0).values
s = (p.values - np.outer(r, c)) / np.sqrt(np.outer(r, c))
u, sv, vt = np.linalg.svd(s, full_matrices=False)
print("share of inertia by axis (%):", (sv ** 2 / (sv ** 2).sum() * 100).round(1)[:3])
print(pd.DataFrame((u * sv)[:, :2] / np.sqrt(r)[:, None], index=counts.index, columns=["axis 1", "axis 2"]).round(3).to_string())

## Images

In [ ]:
# figures for sections 5.1 and 5.2, saved as png files at 300 dpi into this folder
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

ink, ink_secondary, grid_color = "#0b0b0b", "#52514e", "#e5e4df"
country_order = ["Germany", "Spain", "Poland", "Hungary"]
country_color = {"Germany": "#2a78d6", "Spain": "#eb6834", "Poland": "#1baf7a", "Hungary": "#eda100"}
country_marker = {"Germany": "o", "Spain": "s", "Poland": "^", "Hungary": "D"}
outlet_short = {"spiegel": "SP", "welt": "WE", "eldiario": "ED", "elmundo": "EM",
                "polityka": "PO", "rp": "RP", "telex": "TX", "magyar": "MN"}
outlet_name = {"spiegel": "Der Spiegel", "welt": "Die Welt", "eldiario": "elDiario.es", "elmundo": "El Mundo",
               "polityka": "Polityka", "rp": "Rzeczpospolita", "telex": "Telex", "magyar": "Magyar Nemzet"}
legit_color, delegit_color, zero_color, none_color = "#2a78d6", "#e34948", "#b9b8b3", "#f0efec"
blues = LinearSegmentedColormap.from_list("blues", ["#f7fafe", "#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])

plt.rcParams.update({"font.family": "sans-serif", "font.size": 9, "axes.edgecolor": grid_color,
                     "axes.labelcolor": ink_secondary, "xtick.color": ink_secondary, "ytick.color": ink_secondary,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.titlesize": 10,
                     "axes.titleweight": "bold", "axes.titlecolor": ink, "savefig.dpi": 300, "savefig.bbox": "tight"})


def end_labels(ax, x, series, min_gap):
    placed = []
    for country, y in sorted(series.items(), key=lambda item: item[1]):
        y_label = max(y, placed[-1] + min_gap) if placed else y
        placed.append(y_label)
        ax.annotate(country, (x, y), xytext=(x, y_label), textcoords="data", va="center", fontsize=8, color=ink,
                    xycoords="data", annotation_clip=False)


# figure 1: monthly volume of russia-central coverage by country
monthly_volume = pd.read_csv("monthly_volume_by_country.csv", index_col=0)
monthly_volume.index = pd.to_datetime(monthly_volume.index)
fig, ax = plt.subplots(figsize=(7, 3.2))
for country in country_order:
    ax.plot(monthly_volume.index, monthly_volume[country], color=country_color[country], linewidth=1.6, label=country)
ax.set_ylabel("articles per month")
ax.grid(axis="y", color=grid_color, linewidth=0.6)
ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12))
ax.set_title("Russia-central articles per month, by country", loc="left")
fig.savefig("figure_1_monthly_volume.png")
plt.close(fig)

# figure 2: share of articles activating each category, by outlet
heat = category_share[outlets]
fig, ax = plt.subplots(figsize=(7, 5.2))
image = ax.imshow(heat.to_numpy(), cmap=blues, vmin=0, vmax=65, aspect="auto")
ax.set_xticks(range(len(outlets)), [outlet_name[o] for o in outlets], rotation=35, ha="right")
ax.set_yticks(range(len(heat.index)), heat.index)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        value = heat.iat[i, j]
        ax.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=7,
                color="white" if value > 35 else ink)
ax.axhline(6.5, color="white", linewidth=2)
ax.spines[:].set_visible(False)
ax.tick_params(length=0)
colorbar = fig.colorbar(image, ax=ax, fraction=0.03, pad=0.02)
colorbar.set_label("% of articles")
colorbar.outline.set_visible(False)
ax.set_title("Share of articles activating each category", loc="left")
fig.savefig("figure_2_category_heatmap.png")
plt.close(fig)

# figure 3: distribution of article scores by outlet, and the legitimating proportion with its interval
order = outlets[::-1]
parts = [("below zero %", delegit_color, "leans delegitimating"), ("zero %", zero_color, "balanced"),
         ("above zero %", legit_color, "leans legitimating"), ("no activation %", none_color, "no activation")]
fig, (left, right) = plt.subplots(1, 2, figsize=(8, 3.8), gridspec_kw={"width_ratios": [1.6, 1]}, sharey=True)
start = np.zeros(len(order))
for column, color, label in parts:
    values = split.loc[order, column].to_numpy()
    left.barh(range(len(order)), values, left=start, color=color, edgecolor="white", linewidth=1, label=label, height=0.7)
    start += values
left.set_yticks(range(len(order)), [outlet_name[o] for o in order])
left.set_xlim(0, 100)
left.set_xlabel("% of articles")
left.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.18), fontsize=8)
left.set_title("Article scores", loc="left")
for k, o in enumerate(order):
    estimate = outlet_props[o][0] * 100
    low, high = interval(outlet_props[o][1])
    right.plot([low * 100, high * 100], [k, k], color=ink_secondary, linewidth=1.6)
    right.plot(estimate, k, "o", color=ink, markersize=5)
    right.annotate(f"{estimate:.1f}", (high * 100, k), xytext=(5, 0), textcoords="offset points", va="center", fontsize=8, color=ink)
right.set_xlim(0, 55)
right.set_xlabel("% with 95% interval")
right.grid(axis="x", color=grid_color, linewidth=0.6)
right.tick_params(axis="y", length=0)
right.set_title("Legitimating proportion", loc="left")
fig.savefig("figure_3_score_distribution.png")
plt.close(fig)

# figure 4: legitimating proportion by year and country
yearly = share_table(main, "year")[country_order]
fig, ax = plt.subplots(figsize=(6, 3.4))
for country in country_order:
    ax.plot(yearly.index[:-1], yearly[country].iloc[:-1], color=country_color[country], linewidth=1.8,
            marker=country_marker[country], markersize=6, label=country)
    ax.plot(yearly.index[-2:], yearly[country].iloc[-2:], color=country_color[country], linewidth=1.2, linestyle=":")
    ax.plot(yearly.index[-1], yearly[country].iloc[-1], marker=country_marker[country], markersize=6,
            markerfacecolor="white", color=country_color[country])
ax.set_ylim(0, yearly.max().max() * 1.1)
end_labels(ax, yearly.index[-1] + 0.12, yearly.iloc[-1].to_dict(), min_gap=yearly.max().max() * 0.06)
ax.set_xticks(yearly.index, [str(y) if y != yearly.index[-1] else f"{y}\n(Jan to Jun)" for y in yearly.index])
ax.set_ylabel("legitimating proportion, %")
ax.set_xlim(yearly.index[0] - 0.3, yearly.index[-1] + 0.8)
ax.grid(axis="y", color=grid_color, linewidth=0.6)
ax.set_title("Legitimating proportion by year", loc="left")
fig.savefig("figure_4_legitimation_by_year.png")
plt.close(fig)

# figure 5: share of affective and propositional categories in each channel
total_obs, total_res = pooled(observed, resampled)
groups = [("affective", affective), ("propositional", propositional)]
channel_colors = {"text": "#52514e", "image": "#2a78d6"}
fig, ax = plt.subplots(figsize=(5, 3.2))
for g, (label, group) in enumerate(groups):
    for c, channel in enumerate(["text", "image"]):
        estimate = channel_share(total_obs, group, channel) * 100
        low, high = interval(channel_share(total_res, group, channel))
        x = g + (c - 0.5) * 0.36
        ax.bar(x, estimate, width=0.34, color=channel_colors[channel], edgecolor="white", linewidth=1,
               label=channel if g == 0 else None)
        ax.plot([x, x], [low * 100, high * 100], color=ink, linewidth=1)
        ax.annotate(f"{estimate:.1f}", (x, high * 100), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)
ax.set_xticks([0, 1], ["affective categories", "propositional categories"])
ax.set_ylabel("% of the channel's categories")
ax.grid(axis="y", color=grid_color, linewidth=0.6)
ax.set_axisbelow(True)
ax.legend(frameon=False, title=None)
ax.set_title("Category groups in text and images", loc="left")
fig.savefig("figure_5_channels.png")
plt.close(fig)

# figure 7: share of kremlin-friendly parliamentary seats against the legitimating proportion
seats = {"Germany": 16.6, "Spain": 26.3, "Poland": 2.4, "Hungary": 70.9}
fig, ax = plt.subplots(figsize=(5.5, 3.8))
for country in country_order:
    a, b = pairs[country]
    x = seats[country]
    for o in (a, b):
        ax.plot(x, outlet_props[o][0] * 100, marker=country_marker[country], markersize=5,
                markerfacecolor="white", color=country_color[country], linestyle="none")
        ax.annotate(outlet_short[o], (x, outlet_props[o][0] * 100), xytext=(7, 0), textcoords="offset points",
                    va="center", fontsize=7, color=ink_secondary)
    estimate = country_props[country][0] * 100
    low, high = interval(country_props[country][1])
    ax.plot([x, x], [low * 100, high * 100], color=country_color[country], linewidth=2)
    ax.plot(x, estimate, marker=country_marker[country], markersize=8, color=country_color[country], linestyle="none", label=country)
ax.set_xlabel("seats held by parties below the Kremlin_ties midpoint, %")
ax.set_ylabel("legitimating proportion, %")
ax.set_xlim(-2, 80)
ax.set_ylim(0, None)
ax.grid(color=grid_color, linewidth=0.6)
ax.legend(frameon=False, loc="upper left")
ax.annotate("Filled markers: country with 95% interval. Hollow markers: outlets.\nSP Spiegel, WE Welt, ED elDiario.es, EM El Mundo, PO Polityka, RP Rzeczpospolita, TX Telex, MN Magyar Nemzet",
            (0, -0.2), xycoords="axes fraction", fontsize=7, color=ink_secondary, va="top")
ax.set_title("Kremlin-friendly seats and legitimating proportion", loc="left")
fig.savefig("figure_7_seats_and_legitimation.png")
plt.close(fig)

print("saved figure_1, 2, 3, 4, 5 and 7 as png files")